In [ ]:
import torch 
import torch.nn.functional as F
import numpy as np
import os
import pandas as pd
from utils.pad_batch import FEATURE_PADDING_VALUE
from utils.features import getFeatures
from models import AlphaModel, KModel, StateModel
from utils.class_k_model import KModel
from utils.class_state_model import StateModel
import statistics

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_FEATURES=10

print('The model is running on:', DEVICE) 

In [ ]:
# Initialise the models - use your weight file path, otherwise default used
AlphaModel = AlphaModel(num_inputs=NUM_FEATURES).to(DEVICE)
KModel = KModel(num_inputs=NUM_FEATURES).to(DEVICE)
StateModel = StateModel(num_inputs=NUM_FEATURES).to(DEVICE)

AlphaModel.load_state_dict(torch.load("alpha_weights"))
KModel.load_state_dict(torch.load("k_weights"))
StateModel.load_state_dict(torch.load("state_weights"))

AlphaModel.eval()
KModel.eval()
StateModel.eval()

# Predictions on New Tracks

In [4]:
def getPredictions(df, max_size=200):

    features = np.nan_to_num(getFeatures(df["x"].values, df["y"].values), nan=0.0, posinf=0.0, neginf=0.0)
    features = torch.tensor(features, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    length = features.size(1)

    if length < max_size:
        features = F.pad(features, (0, 0, 0, max_size - length), value=FEATURE_PADDING_VALUE)
    elif length > max_size:
        raise ValueError(f"The input size ({length}) is too large. Max allowed size is {max_size}.")


    with torch.no_grad():
        
        pred_alpha_list = AlphaModel(features).cpu().numpy().flatten().squeeze()[:length]
        pred_k_list = KModel(features).cpu().numpy().flatten().squeeze()[:length]
        
        states_log_probs = StateModel(features)
        pred_states_list = torch.argmax(states_log_probs, dim=-1).cpu().numpy().flatten().squeeze()[:length]


    merged_cps, _, _, pred_alpha_list, pred_k_list, pred_states_list = combined_cps(pred_alpha_list, pred_k_list, pred_states_list)

    final_predictions = []
    merged_cps = [0] + merged_cps

    for i in range(len(merged_cps) - 1):
        
        start = merged_cps[i]
        end = merged_cps[i + 1]
        
        log_k_plus1 = np.mean(pred_k_list[start:end])
        final_alpha = np.mean(pred_alpha_list[start:end])
        final_state = statistics.mode(pred_states_list[start:end])

        final_k = 10**log_k_plus1 - 1     

        if final_k < 0.01 and final_alpha < 0.1 or final_state == 0:
            final_state = 0
            final_alpha = 0
            final_k = 0

        elif final_alpha >= 1.9:
            final_state = 3

        final_predictions.append(final_k)
        final_predictions.append(final_alpha)
        final_predictions.append(int(final_state))
        final_predictions.append(end)

    return final_predictions

In [5]:
challenge_data_path = "/home/haidiri/Desktop/andi/challenge_track_2"

N_EXP = 12
N_FOVS = 30
track = 2

path_results = 'res/'
path_track = os.path.join(path_results, f'track_{track}/')

os.makedirs(path_results, exist_ok=True)
os.makedirs(path_track, exist_ok=True)

In [ ]:
for exp in range(N_EXP):
    
    path_exp = os.path.join(path_track, f'exp_{exp}/')
    os.makedirs(path_exp, exist_ok=True)
    
    for fov in range(N_FOVS):

        df = pd.read_csv(challenge_data_path+f'/exp_{exp}/trajs_fov_{fov}.csv')

        traj_idx = df.traj_idx.unique()
        
        submission_file = os.path.join(path_exp, f'fov_{fov}.txt')
        
        with open(submission_file, 'a') as f:
            for idx in traj_idx:
                
                sub_df = df[df.traj_idx == idx]  

                pred =  getPredictions(sub_df)

                prediction_final = [idx.astype(int)] + pred
                formatted_numbers = ','.join(map(str, prediction_final))
                
                f.write(formatted_numbers + '\n')